### Zoteroize and Obsidianize a `Save my Chatbot` Perplexity Dialogue

Replace the citation numbers in a saved Save my Chatbot Perplexity dialogue with matching literature note or zotero item links

In [1]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

In [2]:
tmp_dir = rfw.refwrangle_test_dir / 'tmp'

savemychatbot_perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_multi_prompt_savemychatbot_example.md'
output_file_savemychatbot = tmp_dir / 'tmp_savemychatbot_multiprompt_perplexity_example.md'

##### get the URLs of all parent items in the zotero db, and find out which have obsidian literature notes


In [3]:
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
parentItems = zot.everything(zot.top())

In [4]:
# Make a lookup dict: zotero DB item URL to bibtex citekey
citekeys = {}
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    citekeyThis = rfw.get_citation_key(pdat)

    if 'url' in pdat:
        purl = rfw.normalize_url(pdat['url'])
        if len(purl)>0:
            citekeysForURL[purl].append(citekeyThis)

repeatedURLs = {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}

if (nURLrepeats := len(repeatedURLs)) > 0:
    print(f"There were {nURLrepeats} URLs with > 1 parent (citekey)")
    for url in repeatedURLs.keys():
        print(f"{repeatedURLs[url]}\n\t{url}")
    raise Exception(f'Not built for repeated URLS: {nURLrepeats=}.')

url_to_citekey={}
for (key, value_list) in citekeysForURL.items():
    url_to_citekey[key] = value_list[0]

In [5]:
# Collect info about each zotero DB item that has a URL
lit_note_file_stems = {fNm.stem for fNm in rfw.lit_notes_obsidian_dir.glob('*.md')}

zot_db_items = []
citekeysForURL = defaultdict(list)
for parent in parentItems:
    pdat = parent['data']
    if not (title := pdat.get('title')):
        continue

    citekeyThis = rfw.get_citation_key(pdat)
    zot_db_items.append(dict(citekey=citekeyThis, zotkey=parent['key'], title=title, hasLitNote=citekeyThis in lit_note_file_stems))

    if url := pdat.get('url'):
        if normalized_url :=rfw.normalize_url(url):
            citekeysForURL[normalized_url].append(citekeyThis)

if repeatedURLs := {url:citekeysForURL[url] for url in citekeysForURL.keys() if len(citekeysForURL[url])>1}:
    print(f"Found {len(repeatedURLs)} URLs with > 1 parent (citekey)")
    for url, citekeys in repeatedURLs.items():
        print(f"{', '.join(citekeys)}\n\t{url}")
    raise Exception(f'Not built for repeated URLS')

citekey_to_url = {citekeys[0]: url for url, citekeys in citekeysForURL.items()}
url_to_citekey = {url: citekey for citekey, url in citekey_to_url.items()}

zot_db_items = pd.DataFrame(zot_db_items).set_index('citekey')
zot_db_items['url'] = pd.Series(citekey_to_url)
zot_db_items = zot_db_items.reset_index()

if sum(hasNoURL := zot_db_items.url.isna()):
    print(f"Dropping {sum(hasNoURL)} of {len(zot_db_items)} zotero entries with no URL:")
    display((zot_db_items_no_url := zot_db_items[hasNoURL]).head())
    zot_db_items = zot_db_items[~hasNoURL]


Dropping 149 of 1681 zotero entries with no URL:


,citekey,zotkey,title,hasLitNote,url
1,MMSDataModelSummary_v5.2,8XJHRYMU,MMS Data Model Package Summary v5.2,False,NaN
16,Seals99irradFrcstDiag,WYP9J7EU,The heart of suny irradiance forecasting,False,NaN
92,LaPaglia13TestIncrsSuggestibility,LDTF7M3L,Testing increases suggestibility for narrative...,False,NaN
153,Gaur20attribModellingRvw,HLHKVCLX,Attribution modelling in marketing: Literature...,False,NaN
200,Wang19predOptcoolLdFrcst,R7TJLE7Y,Cooling load forecasting-based predictive opti...,False,NaN


## For markdown from the "Save my Chatbot" chrome/firefox extension

In [6]:
import re
import pandas as pd
from typing import Optional, Dict, List, Tuple
from collections import Counter

def zotero_item_link(zotero_item_key, link_text):
    return f'[{link_text}](zotero://select/library/items/{zotero_item_key})'

def find_zotero_item_by_url(url: str, zot_db_items: pd.DataFrame) -> Optional[Dict]:
    """Find a Zotero item by its URL."""
    normalized_url = rfw.normalize_url(url)
    matches = zot_db_items[zot_db_items['url'] == normalized_url]
    if not matches.empty:
        return matches.iloc[0].to_dict()  # Return as a dictionary
    return None

def find_zotero_item_by_title(target_title: str, zot_db_items: pd.DataFrame) -> Optional[Dict]:
    """Find the Zotero item with the best matching title."""
    zotero_items = zot_db_items.to_dict('records')
    best_match_item = None
    best_score = 0

    for item in zotero_items:
        score = rfw.match_titles(target_title, item['title'], main_title_only=False)
        if score > best_score:
            best_match_item = item.copy()
            best_score = score

    if best_score > 95: # max==100: stringnet, limit false matches
        # ic(best_score, target_title, best_match_item['title'])
        return best_match_item 
    
    return None

def build_source_url_to_title(sources_content: str) -> Dict[str, str]:
    """Build a dictionary mapping URLs to titles from the sources section."""
    source_url_to_title = {}
    matches = re.findall(r'- \[(.*?)\]\((https?://\S+)\)', sources_content)
    for title, url in matches:
        normalized_url = rfw.normalize_url(url)
        title = re.sub(r'^\s*\(\d+\)\s*', '', title) # remove ref num
        source_url_to_title[normalized_url] = title.strip()
    return source_url_to_title

def replace_links_with_zotero_items(
    body_content: str,
    sources_content: str,
    zot_db_items: pd.DataFrame,
) -> Tuple[str, str, Counter]:
    """
    Replace links in body content and sources content with Zotero links or leave them as-is.
    
    Returns:
        - Updated body content.
        - Updated sources content.
        - A Counter of URLs in the body that were not found in the sources."""
    
    # Build source URL-to-title mapping
    source_url_to_title = build_source_url_to_title(sources_content)

    missing_sources_links = Counter()
    body_link_num_not_in_zotero = {}

    def link_to_obsidian_or_zotero(zotero_item):
        """Returns link to Obsidian lit note if it exists, else to zotero item"""
        if zotero_item.get('hasLitNote', False):
            return f'[[{zotero_item["citekey"]}]]'
        else:
            link_text = f'{zotero_item["citekey"]}\u2794{zotero_item["zotkey"]}'
            return zotero_item_link(zotero_item["zotkey"], link_text)

    def make_my_lit_link(url):
        """If a zotero item has a matching url, or title that matches a source's 
        section link title, then return a link to that item or its obsidian note."""

        if zotero_item := find_zotero_item_by_url(url, zot_db_items):
            return link_to_obsidian_or_zotero(zotero_item)

        if url in source_url_to_title:
            # try to replace matching source link title with zotero item title
            title = source_url_to_title[url]
            if zotero_item := find_zotero_item_by_title(title, zot_db_items):
                return link_to_obsidian_or_zotero(zotero_item)

        return None # no kind of zotero item match
            
    def swap_my_lit_link_body(doc_match):
        """Replace a body section link with one pointing to zotero/obsidian, if possible.
        Otherwise bold the link so it's clear there was no match
        
        Arg: doc_match: a regexp match object to a documenent body section link"""
         
        url_body_link = rfw.normalize_url(doc_match.group(2))
        body_link_num = doc_match.group(1)
        
        # ic(url_body_link, body_link_num)
        
        if url_body_link not in source_url_to_title:
            missing_sources_links[url_body_link] += 1 # for later error reporting

        if my_link := make_my_lit_link(url_body_link):
            return my_link

        # a link in body that wasn't in zotero: emphasize it in both the body and the sources
        body_link_num_not_in_zotero[body_link_num] = True
        return f'**{doc_match.group(0)}**'

    def append_my_lit_link_source(doc_match):
        """Append a sources section link with a highlighted link pointing to zotero/obsidian, if possible.
        
        Arg: doc_match: a regexp match object to a document sources section link"""

        #number_with_parentheses = f"**({match.group(1)})**"  # Surround the entire parenthesized number with '**'
        # text = match.group(2) + " bob"                      # Append 'bob' to the link text
        # url = match.group(3)                                # Keep the URL unchanged


        url_source_link = rfw.normalize_url(doc_match.group(3))
        # url_source_link = rfw.normalize_url(doc_match.group(2))

        source_link_num = doc_match.group(1)
        output_link_num = f'({source_link_num})'
        descript_source_link = doc_match.group(2)
        if my_link := make_my_lit_link(url_source_link):
            # descript_source_link = doc_match.group(1)
            return f'[{output_link_num} {descript_source_link}]({url_source_link}) =={my_link}=='
            # return f'- [{descript_source_link}]({url_source_link}) =={my_link}=='

        #return doc_match.group(0)
        if body_link_num_not_in_zotero.get(source_link_num):
            output_link_num = f'**{output_link_num}**' # bold it, to match body appearance
        
        return f'[{output_link_num} {descript_source_link}]({url_source_link})'

    updated_body_content = re.sub(r'\[(.*?)\]\((https?://\S+)\)', swap_my_lit_link_body, body_content)

    updated_sources_content = re.sub(r'\[\((\d+)\)\s*(.*?)\]\((https?://\S+)\)', append_my_lit_link_source, sources_content)
    # updated_sources_content = re.sub(r'- \[(.*?)\]\((https?://\S+)\)', append_my_lit_link, sources_content)

    # ic(body_link_num_not_in_zotero)
    
    return updated_body_content, updated_sources_content, missing_sources_links

def relink_perplexity_export_SmC(input_file: str, output_file: str, zot_db_items: pd.DataFrame):
    """Process a markdown file to replace links with Zotero references."""
    
    with open(input_file, 'r') as infile:
        content = infile.read()

    # Split content into sections based on level 2 headers named "User"
    sections = re.split(r'(?<=\n)## User', content)

    processed_sections = [sections[0]]  # start with header
    log_missing_links = []

    for section_idx, section in enumerate(sections[1:], start=1):  # Skip anything before the first "User" section
        # Split each section into body and sources parts
        parts = re.split(r'(\n---\s*\n\s*\*\*Sources:\*\*\s*\n)', section)
        
        if len(parts) < 3:
            print('\nSkipping section without both body and sources:\n')
            ic(parts)
            continue

        body_content = parts[0]
        
        # Process body and sources content to replace links with Zotero references
        updated_body_content, updated_sources_content, missing_sources_links_counter = replace_links_with_zotero_items(
            body_content,
            parts[2],
            zot_db_items,
        )

        if missing_sources_links_counter:
            log_missing_links.append(
                f"Section {section_idx}: Missing links - " +
                ", ".join([f"{url} (count: {count})" for url, count in missing_sources_links_counter.items()])
            )

        processed_sections.append(f"## User{updated_body_content}")
        processed_sections.append(parts[1])  # Sources header remains unchanged
        processed_sections.append(updated_sources_content)

    with open(output_file, 'w') as outfile:
        outfile.write(''.join(processed_sections))

    if log_missing_links:
        print("Log of missing links:")
        for log_entry in log_missing_links:
            print(log_entry)

' # Example usage\nif __name__ == "__main__":\n    input_markdown_file = "example.md"\n    output_markdown_file = "output.md"\n\n    # Example Zotero database (as a Pandas DataFrame)\n    zot_db_items_df = pd.DataFrame([\n        {\'url\': \'https://example.com\', \'title\': \'Example Title\', \'citekey\': \'ExampleCiteKey\', \'zotkey\': \'Z12345\', \'hasLitNote\': True},\n        {\'url\': \'\', \'title\': \'Another Example Title\', \'citekey\': \'AnotherCiteKey\', \'zotkey\': \'\n '

In [10]:

ic(savemychatbot_perplexity_dialog_file, output_file_savemychatbot)
relink_perplexity_export_SmC(savemychatbot_perplexity_dialog_file, output_file_savemychatbot, zot_db_items)
print('Done.')

ic| savemychatbot_perplexity_dialog_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/perplexity_multi_prompt_savemychatbot_example.md')
    output_file_savemychatbot: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_savemychatbot_multiprompt_perplexity_example.md')


Log of missing links:
Section 3: Missing links - https://en.wikipedia.org/wiki/total_correlation (count: 4), https://arxiv.org/abs/2011.04794 (count: 3), https://proceedings.mlr.press/v206/bai23a/bai23a.pdf (count: 4)
Done.
